Enrollment No: 240840116029

Practical 10 :Mini Project: Analyze a real-world dataset, build an interactive dashboard, and present actionable insights with ethical considerations

## Cell 1 — Install required libraries
Run this cell first.

In [ ]:
!pip -q install dash plotly pandas numpy

## Cell 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from google.colab import files
from dash import Dash, dcc, html, Input, Output
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


## Cell 3 — Upload a hotel booking dataset

In [ ]:
UPLOAD_REAL_CSV = False  # Change to True to upload your own CSV

uploaded_df = None
if UPLOAD_REAL_CSV:
    uploaded = files.upload()
    if uploaded:
        file_name = next(iter(uploaded))
        uploaded_df = pd.read_csv(file_name)
        print("Uploaded:", file_name)
        display(uploaded_df.head())
else:
    print("No CSV uploaded. Simulated practice data will be created in Cell 4.")

No CSV uploaded. Simulated practice data will be created in Cell 4.


## Cell 4 — Create a simulated practice dataset if no CSV was uploaded

In [ ]:
if uploaded_df is None:
    np.random.seed(21)
    n = 800
    booking_dates = pd.to_datetime(
        np.random.choice(pd.date_range("2024-01-01", "2025-12-31", freq="D"), n)
    )
    room_types = ["Single", "Double", "Deluxe", "Suite"]
    room_prices = {"Single": 1800, "Double": 2800, "Deluxe": 4200, "Suite": 6500}

    hotel_type = np.random.choice(["City Hotel", "Resort Hotel"], n, p=[0.65, 0.35])
    room_type = np.random.choice(room_types, n, p=[0.20, 0.40, 0.28, 0.12])
    nights = np.random.randint(1, 8, n)
    guests = np.random.choice([1, 2, 3, 4], n, p=[0.18, 0.52, 0.20, 0.10])
    price = np.array([room_prices[r] for r in room_type])
    status = np.random.choice(["Confirmed", "Cancelled", "Pending"], n,
                              p=[0.72, 0.18, 0.10])
    checkin = booking_dates + pd.to_timedelta(np.random.randint(1, 90, n), unit="D")
    checkout = checkin + pd.to_timedelta(nights, unit="D")

    df = pd.DataFrame({
        "Booking_ID": [f"HB{i:05d}" for i in range(1, n + 1)],
        "Hotel_Type": hotel_type,
        "Customer_Type": np.random.choice(["New", "Returning", "Corporate", "Family"], n,
                                          p=[0.40, 0.30, 0.18, 0.12]),
        "Booking_Date": booking_dates,
        "Check_In_Date": checkin,
        "Check_Out_Date": checkout,
        "Room_Type": room_type,
        "Number_of_Guests": guests,
        "Number_of_Nights": nights,
        "Room_Price": price,
        "Total_Revenue": price * nights,
        "Booking_Status": status,
        "Country": np.random.choice(["India", "United Kingdom", "USA", "UAE", "Germany", "France"], n)
    })
    DATA_SOURCE = "Simulated practice data — not real hotel records"
else:
    df = uploaded_df.copy()
    DATA_SOURCE = "User-uploaded CSV — verify and document its original source"

print("Data source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
display(df.head())

Data source: Simulated practice data — not real hotel records
Dataset shape: (800, 13)


,Booking_ID,Hotel_Type,Customer_Type,Booking_Date,Check_In_Date,Check_Out_Date,Room_Type,Number_of_Guests,Number_of_Nights,Room_Price,Total_Revenue,Booking_Status,Country
0,HB00001,City Hotel,Returning,2024-07-26,2024-09-03,2024-09-06,Single,2,3,1800,5400,Cancelled,France
1,HB00002,Resort Hotel,Returning,2024-02-18,2024-04-18,2024-04-22,Deluxe,2,4,4200,16800,Pending,UAE
2,HB00003,City Hotel,Corporate,2025-01-03,2025-01-15,2025-01-20,Deluxe,1,5,4200,21000,Pending,India
3,HB00004,City Hotel,Returning,2024-04-30,2024-07-17,2024-07-18,Single,3,1,1800,1800,Confirmed,UAE
4,HB00005,City Hotel,Returning,2025-02-23,2025-03-05,2025-03-10,Single,4,5,1800,9000,Confirmed,USA


## Cell 5 — Standardize column names and check required fields

In [ ]:
# Example: rename your source columns to the standard names used in this notebook.
rename_map = {
    # "hotel": "Hotel_Type",
    # "market_segment": "Customer_Type",
    # "reservation_status": "Booking_Status",
    # "reserved_room_type": "Room_Type",
    # "country": "Country",
    # "adr": "Room_Price",
}
df = df.rename(columns=rename_map)

required = ["Booking_Status", "Room_Type", "Customer_Type", "Booking_Date"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Edit rename_map or prepare these fields in your CSV before continuing."
    )

if "Booking_ID" not in df.columns:
    df["Booking_ID"] = [f"ROW{i:05d}" for i in range(1, len(df) + 1)]
if "Hotel_Type" not in df.columns:
    df["Hotel_Type"] = "Not specified"
if "Country" not in df.columns:
    df["Country"] = "Not specified"
if "Number_of_Guests" not in df.columns:
    df["Number_of_Guests"] = np.nan
if "Number_of_Nights" not in df.columns:
    df["Number_of_Nights"] = np.nan
if "Room_Price" not in df.columns:
    df["Room_Price"] = np.nan
if "Total_Revenue" not in df.columns:
    if {"Number_of_Nights", "Room_Price"}.issubset(df.columns):
        df["Total_Revenue"] = df["Number_of_Nights"] * df["Room_Price"]
    else:
        df["Total_Revenue"] = np.nan

print("Column validation completed.")
print(df.columns.tolist())

Column validation completed.
['Booking_ID', 'Hotel_Type', 'Customer_Type', 'Booking_Date', 'Check_In_Date', 'Check_Out_Date', 'Room_Type', 'Number_of_Guests', 'Number_of_Nights', 'Room_Price', 'Total_Revenue', 'Booking_Status', 'Country']


## Cell 6 — Clean the dataset

In [ ]:
date_cols = ["Booking_Date", "Check_In_Date", "Check_Out_Date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = ["Number_of_Guests", "Number_of_Nights", "Room_Price", "Total_Revenue"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["Booking_Date", "Booking_Status", "Room_Type", "Customer_Type"])
df["Booking_Status"] = df["Booking_Status"].astype(str).str.strip().str.title()
df["Room_Type"] = df["Room_Type"].astype(str).str.strip().str.title()
df["Customer_Type"] = df["Customer_Type"].astype(str).str.strip().str.title()
df["Month"] = df["Booking_Date"].dt.to_period("M").astype(str)

print("Duplicate rows removed:", before - len(df))
print("Rows after cleaning:", len(df))
print("Missing values:")
display(df.isna().sum().to_frame("Missing values"))

Duplicate rows removed: 0
Rows after cleaning: 800
Missing values:


,Missing values
Booking_ID,0
Hotel_Type,0
Customer_Type,0
Booking_Date,0
Check_In_Date,0
Check_Out_Date,0
Room_Type,0
Number_of_Guests,0
Number_of_Nights,0
Room_Price,0


## Cell 7 — Calculate hotel performance measures

In [ ]:
# Revenue can be analyzed only where it is present and valid.
df["Total_Revenue"] = df["Total_Revenue"].clip(lower=0)
df["Number_of_Nights"] = df["Number_of_Nights"].clip(lower=0)

# Booking cancellation percentage is based on the rows in the selected dataset.
cancelled_count = (df["Booking_Status"] == "Cancelled").sum()
cancellation_rate = cancelled_count / len(df) * 100 if len(df) else 0

# Average daily rate (ADR) is available where revenue and nights are known.
valid_adr = df[(df["Number_of_Nights"] > 0) & df["Total_Revenue"].notna()]
average_daily_rate = (
    valid_adr["Total_Revenue"].sum() / valid_adr["Number_of_Nights"].sum()
    if len(valid_adr) else np.nan
)

print(f"Cancellation rate: {cancellation_rate:.1f}%")
print("Average daily rate:", f"₹{average_daily_rate:,.2f}" if pd.notna(average_daily_rate) else "Not available")
display(df.head())

Cancellation rate: 17.5%
Average daily rate: ₹3,401.63


,Booking_ID,Hotel_Type,Customer_Type,Booking_Date,Check_In_Date,Check_Out_Date,Room_Type,Number_of_Guests,Number_of_Nights,Room_Price,Total_Revenue,Booking_Status,Country,Month
0,HB00001,City Hotel,Returning,2024-07-26,2024-09-03,2024-09-06,Single,2,3,1800,5400,Cancelled,France,2024-07
1,HB00002,Resort Hotel,Returning,2024-02-18,2024-04-18,2024-04-22,Deluxe,2,4,4200,16800,Pending,UAE,2024-02
2,HB00003,City Hotel,Corporate,2025-01-03,2025-01-15,2025-01-20,Deluxe,1,5,4200,21000,Pending,India,2025-01
3,HB00004,City Hotel,Returning,2024-04-30,2024-07-17,2024-07-18,Single,3,1,1800,1800,Confirmed,UAE,2024-04
4,HB00005,City Hotel,Returning,2025-02-23,2025-03-05,2025-03-10,Single,4,5,1800,9000,Confirmed,USA,2025-02


## Cell 8 — Exploratory Data Analysis (EDA)

In [ ]:
print("Dataset overview")
display(df.describe(include="all").T)

print("\nBooking status counts")
display(df["Booking_Status"].value_counts().to_frame("Bookings"))

print("\nRoom type counts")
display(df["Room_Type"].value_counts().to_frame("Bookings"))

print("\nCustomer type counts")
display(df["Customer_Type"].value_counts().to_frame("Bookings"))

Dataset overview


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Booking_ID,800,800,HB00784,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hotel_Type,800,2,City Hotel,530,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Type,800,4,New,309,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Booking_Date,800,NaN,NaN,NaN,2024-12-24 05:36:36.000000256,2024-01-01 00:00:00,2024-07-05 00:00:00,2024-12-14 00:00:00,2025-06-14 00:00:00,2025-12-31 00:00:00,NaN
Check_In_Date,800,NaN,NaN,NaN,2025-02-06 12:45:00,2024-01-04 00:00:00,2024-08-24 00:00:00,2025-01-28 12:00:00,2025-07-23 12:00:00,2026-03-14 00:00:00,NaN
Check_Out_Date,800,NaN,NaN,NaN,2025-02-10 12:16:12,2024-01-05 00:00:00,2024-08-25 18:00:00,2025-01-31 12:00:00,2025-07-25 12:00:00,2026-03-21 00:00:00,NaN
Room_Type,800,4,Double,311,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Number_of_Guests,800.0,NaN,NaN,NaN,2.225,1.0,2.0,2.0,3.0,4.0,0.845736
Number_of_Nights,800.0,NaN,NaN,NaN,3.98,1.0,2.0,4.0,6.0,7.0,2.011133
Room_Price,800.0,NaN,NaN,NaN,3419.0,1800.0,2800.0,2800.0,4200.0,6500.0,1411.685188



Booking status counts


,Bookings
Booking_Status,
Confirmed,579
Cancelled,140
Pending,81



Room type counts


,Bookings
Room_Type,
Double,311
Deluxe,226
Single,169
Suite,94



Customer type counts


,Bookings
Customer_Type,
New,309
Returning,242
Corporate,149
Family,100


## Cell 9 — Create summary tables

In [ ]:
status_summary = df.groupby("Booking_Status", as_index=False).size().rename(columns={"size": "Bookings"})
room_summary = df.groupby("Room_Type", as_index=False).size().rename(columns={"size": "Bookings"})
customer_summary = df.groupby("Customer_Type", as_index=False).size().rename(columns={"size": "Bookings"})
monthly_summary = (
    df.groupby("Month", as_index=False)
      .agg(Bookings=("Booking_ID", "count"),
           Revenue=("Total_Revenue", "sum"))
      .sort_values("Month")
)
display(status_summary)
display(room_summary)
display(customer_summary)
display(monthly_summary.head())

,Booking_Status,Bookings
0,Cancelled,140
1,Confirmed,579
2,Pending,81


,Room_Type,Bookings
0,Deluxe,226
1,Double,311
2,Single,169
3,Suite,94


,Customer_Type,Bookings
0,Corporate,149
1,Family,100
2,New,309
3,Returning,242


,Month,Bookings,Revenue
0,2024-01,31,422900
1,2024-02,25,358300
2,2024-03,37,525100
3,2024-04,37,482300
4,2024-05,28,290000


## Cell 10 — Create four charts for initial analysis

In [ ]:
fig1 = px.pie(status_summary, names="Booking_Status", values="Bookings",
               title="Booking Status Distribution", hole=0.35)
fig1.show()

fig2 = px.line(monthly_summary, x="Month", y="Revenue", markers=True,
               title="Monthly Hotel Revenue")
fig2.update_layout(xaxis_tickangle=-45, yaxis_title="Revenue (₹)")
fig2.show()

fig3 = px.bar(room_summary, x="Room_Type", y="Bookings",
              title="Bookings by Room Type")
fig3.show()

fig4 = px.pie(customer_summary, names="Customer_Type", values="Bookings",
               title="Customer Type Distribution", hole=0.35)
fig4.show()

## Cell 11 — Build the interactive Dash dashboard

Select hotel type to filter the charts. The four charts update using the selected hotel type. This notebook uses the external display mode supported by Dash in Colab.

In [ ]:
app = Dash(__name__)
hotel_options = [{"label": "All Hotels", "value": "All"}] + [
    {"label": str(x), "value": str(x)} for x in sorted(df["Hotel_Type"].dropna().unique())
]

app.layout = html.Div([
    html.H1("Hotel Management Analytics Dashboard",
            style={"textAlign": "center", "color": "#17324D"}),
    html.P("Explore bookings, revenue, room preferences and customer segments.",
           style={"textAlign": "center"}),
    html.Div([
        html.Label("Select Hotel Type:"),
        dcc.Dropdown(id="hotel-filter", options=hotel_options, value="All",
                     clearable=False, style={"color": "#222", "maxWidth": "500px"})
    ], style={"padding": "15px"}),
    html.Div(id="hotel-kpis",
             style={"display": "flex", "gap": "15px", "flexWrap": "wrap",
                    "justifyContent": "space-around"}),
    html.Div([
        dcc.Graph(id="booking-status-chart"),
        dcc.Graph(id="monthly-revenue-chart"),
        dcc.Graph(id="room-type-chart"),
        dcc.Graph(id="customer-type-chart")
    ], style={"display": "grid",
              "gridTemplateColumns": "repeat(auto-fit, minmax(350px, 1fr))",
              "gap": "12px", "padding": "10px"})
], style={"fontFamily": "Arial", "backgroundColor": "#F4F7FA", "padding": "15px"})

@app.callback(
    Output("hotel-kpis", "children"),
    Output("booking-status-chart", "figure"),
    Output("monthly-revenue-chart", "figure"),
    Output("room-type-chart", "figure"),
    Output("customer-type-chart", "figure"),
    Input("hotel-filter", "value")
)
def update_hotel_dashboard(selected_hotel):
    filtered = df if selected_hotel == "All" else df[df["Hotel_Type"] == selected_hotel]

    total_bookings = len(filtered)
    total_revenue = filtered["Total_Revenue"].sum(min_count=1)
    cancelled = (filtered["Booking_Status"] == "Cancelled").sum()
    cancel_pct = cancelled / len(filtered) * 100 if len(filtered) else 0
    revenue_text = "N/A" if pd.isna(total_revenue) else f"₹{total_revenue:,.0f}"

    def kpi(title, value):
        return html.Div([
            html.H4(title, style={"marginBottom": "5px"}),
            html.H2(value, style={"marginTop": "0", "color": "#176B87"})
        ], style={"background": "white", "padding": "15px",
                  "borderRadius": "10px", "minWidth": "180px",
                  "boxShadow": "0 1px 5px #D5DDE5"})

    cards = [
        kpi("Total Bookings", f"{total_bookings:,}"),
        kpi("Total Revenue", revenue_text),
        kpi("Cancellation Rate", f"{cancel_pct:.1f}%"),
        kpi("Room Types", f"{filtered['Room_Type'].nunique():,}")
    ]

    status = filtered.groupby("Booking_Status", as_index=False).size().rename(columns={"size": "Bookings"})
    f1 = px.pie(status, names="Booking_Status", values="Bookings",
                title="Booking Status Distribution", hole=0.35)

    month = filtered.assign(Month=filtered["Booking_Date"].dt.to_period("M").astype(str))
    month = month.groupby("Month", as_index=False)["Total_Revenue"].sum().sort_values("Month")
    f2 = px.line(month, x="Month", y="Total_Revenue", markers=True,
                 title="Monthly Hotel Revenue")
    f2.update_layout(xaxis_tickangle=-45, yaxis_title="Revenue (₹)")

    room = filtered.groupby("Room_Type", as_index=False).size().rename(columns={"size": "Bookings"})
    f3 = px.bar(room, x="Room_Type", y="Bookings", title="Bookings by Room Type")

    customer = filtered.groupby("Customer_Type", as_index=False).size().rename(columns={"size": "Bookings"})
    f4 = px.pie(customer, names="Customer_Type", values="Bookings",
                title="Customer Type Distribution", hole=0.35)

    for fig in [f1, f2, f3, f4]:
        fig.update_layout(template="plotly_white", margin=dict(l=30, r=20, t=60, b=40))

    return cards, f1, f2, f3, f4

print("Hotel dashboard is ready.")

Hotel dashboard is ready.


## Cell 12 — Run the dashboard
Run this cell and open the external dashboard link shown by Colab.

In [ ]:
app.run(jupyter_mode="external", debug=False, port=8050)

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Cell 13 — Generate actionable insights


In [ ]:
if len(df):
    top_room = df["Room_Type"].value_counts().idxmax()
    top_customer = df["Customer_Type"].value_counts().idxmax()
    top_status = df["Booking_Status"].value_counts().idxmax()

    monthly_revenue = df.groupby("Month")["Total_Revenue"].sum(min_count=1).dropna()
    best_revenue_month = monthly_revenue.idxmax() if len(monthly_revenue) else "Not available"

    cancel_count = (df["Booking_Status"] == "Cancelled").sum()
    cancel_pct = cancel_count / len(df) * 100

    print("ACTIONABLE INSIGHTS")
    print(f"1. Most frequently booked room type: {top_room}")
    print(f"2. Most common customer type: {top_customer}")
    print(f"3. Most common booking status: {top_status}")
    print(f"4. Highest-revenue booking month: {best_revenue_month}")
    print(f"5. Cancellation rate: {cancel_pct:.1f}%")
    print("\nSuggested actions:")
    print("- Review demand for popular room types when planning room inventory.")
    print("- Study cancellation patterns and booking lead times before changing policies.")
    print("- Plan promotions during lower-revenue months after checking seasonality.")
    print("- Compare customer segments without using sensitive traits or unfair assumptions.")
else:
    print("No records available for insights.")

ACTIONABLE INSIGHTS
1. Most frequently booked room type: Double
2. Most common customer type: New
3. Most common booking status: Confirmed
4. Highest-revenue booking month: 2025-05
5. Cancellation rate: 17.5%

Suggested actions:
- Review demand for popular room types when planning room inventory.
- Study cancellation patterns and booking lead times before changing policies.
- Plan promotions during lower-revenue months after checking seasonality.
- Compare customer segments without using sensitive traits or unfair assumptions.


## Cell 14 — Ethical considerations and limitations

Include these points in the report:

1. **Guest privacy:** Remove names, phone numbers, email addresses, payment details and other personal identifiers.
2. **Data security:** Do not publish guest-level booking records in a public dashboard.
3. **Fair treatment:** Avoid discriminatory pricing or decisions based on protected or sensitive characteristics.
4. **Data quality:** Document missing values, duplicate bookings, date errors and assumptions.
5. **Transparency:** State whether the dataset is public, collected with permission, or simulated.
6. **Responsible interpretation:** A booking is not necessarily a completed stay; cancellation and no-show definitions must be checked.

**Limitations:** Revenue is not profit. Without operating costs, taxes, refunds and room inventory availability, this dataset cannot calculate profit or a reliable occupancy rate. A true occupancy rate requires rooms available and rooms actually occupied over a defined period.

## Cell 15 — Export cleaned data and summaries

In [ ]:
df.to_csv("cleaned_hotel_booking_data.csv", index=False)
status_summary.to_csv("hotel_booking_status_summary.csv", index=False)
room_summary.to_csv("hotel_room_type_summary.csv", index=False)
monthly_summary.to_csv("hotel_monthly_revenue_summary.csv", index=False)
print("Exported cleaned data and summary CSV files.")
files.download("cleaned_hotel_booking_data.csv")

Exported cleaned data and summary CSV files.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This mini-project demonstrates how hotel booking data can be cleaned, analyzed and presented through an interactive dashboard. It helps hotel managers monitor booking status, revenue trends, room demand and customer segments to support operational planning.

**Before final submission:** If your assignment requires real-world data, replace the simulated dataset with a documented public dataset. Record the source, license, access date, data dictionary and preprocessing steps.